# Game 1 — UST Eco Route Collector

This notebook rewrites the project setup into a documented `.ipynb` format for Deliverable 1.


In [1]:
import pandas as pd
import heapq
# import random

## Load datasets


In [2]:
game_nodes = pd.read_csv("ust_selected_game_nodes.csv")
building_edges = pd.read_csv("ust_building_edges.csv")
energy_assets = pd.read_csv("ust_energy_assets.csv")

display(game_nodes.head())
display(building_edges.head())
display(energy_assets.head())



,building_name,code,map_number,category,sustainability_relevance,source_url,status
0,Anderson Student Center,ASC,28,student_center,LEED Gold; major campus hub,https://www.stthomas.edu/_media-library/_docum...,confirmed_on_map
1,Anderson Parking Facility,APF,65,parking,EV charging; secure bike storage,https://www.stthomas.edu/_media-library/_docum...,confirmed_on_map
2,Binz Refectory,BIN,67,dining,Compost-collection milestone in sustainability...,https://www.stthomas.edu/_media-library/_docum...,confirmed_on_map
3,Brady Educational Center,BEC,68,academic,Near Brady Hall solar milestone in sustainabil...,https://www.stthomas.edu/_media-library/_docum...,confirmed_on_map
4,Center for Microgrid Research / Greenhouse,NaN,61,energy_research,Microgrid expansion connects up to 9 buildings,https://www.stthomas.edu/_media-library/_docum...,confirmed_on_map


,Source,Target,Distance
0,ARC,FieldHouse,0.000539
1,ARC,Track,0.001118
2,Admissions,MusicBuilding,0.000943
3,Anderson,KochCommons,0.000447
4,Anderson,StudentCenterAnnex,0.000707


,asset_type,location_name,designation_or_feature,year_or_opening,asset_role,source_url,status
0,leed_building,Anderson Student Center,LEED Gold,NaN,green building,https://www.stthomas.edu/about/sustainability/,confirmed
1,leed_building,Frey Residence Hall,LEED Platinum,2020.0,green building,https://www.stthomas.edu/residence-life/polici...,confirmed
2,leed_building,Schoenecker Center,LEED Gold,2024.0,green building,https://www.stthomas.edu/about/sustainability/,confirmed
3,leed_building,Schoenecker Hall North,LEED Silver,NaN,green building,https://www.stthomas.edu/about/sustainability/,confirmed
4,solar_asset,Brady Hall,First solar photovoltaic panels installed on c...,2012.0,solar generation milestone,https://look.stthomas.edu/view/938809190,confirmed


## Read building variables


In [3]:
variables = []

for row in game_nodes.itertuples():
    variables.append(row.building_name)

print(variables)



['Anderson Student Center', 'Anderson Parking Facility', 'Binz Refectory', 'Brady Educational Center', 'Center for Microgrid Research / Greenhouse', 'Frey Residence Hall', 'Murray-Herrick Campus Center', "O'Shaughnessy Educational Center", "O'Shaughnessy-Frey Library Center", "O'Shaughnessy Science Hall", 'Owens Science Hall', "O'Shaughnessy Stadium", 'Schoenecker Center', 'Schoenecker Hall North', 'Center for Well-Being', 'Archbishop Ireland Memorial Library']


## Create domains


Create an empty dictionary to store domains for each building  
Variable = building  
Domain = possible sustainability assignments for that building

In [4]:
domains = {}

# Initialize every building with an empty list
# We will fill these lists in later depending on building type
for building in variables:
    domains[building] = []

# Loop through the energy assets dataset
# This dataset contains sustainability and LEED information
for row in energy_assets.itertuples():

    # If the building is LEED certified:
    # - LEED buildings keep their LEED designation
    # - They are not allowed to receive additional sustainability upgrades
    if ("LEED" in str(row.designation_or_feature)
        and row.location_name in domains):

        domains[row.location_name] = [row.designation_or_feature]

    # The greenhouse/microgrid building is already sustainability-focused,
    # so we force it to receive an "energy audit" role
    if row.location_name == 'Center for Microgrid Research / Greenhouse':
        domains[row.location_name] = ['energy audit']

# For all remaining buildings that still have empty domains:
# Give them the standard sustainability options
for building in domains:

    if not domains.get(building):
        # recycling = recycling hub
        # compost = compost collection site
        # none = no sustainability upgrade assigned
        domains[building] = [
            'recycling',
            'compost',
            'none'
        ]

print(domains)



{'Anderson Student Center': ['LEED Gold'], 'Anderson Parking Facility': ['recycling', 'compost', 'none'], 'Binz Refectory': ['recycling', 'compost', 'none'], 'Brady Educational Center': ['recycling', 'compost', 'none'], 'Center for Microgrid Research / Greenhouse': ['energy audit'], 'Frey Residence Hall': ['LEED Platinum'], 'Murray-Herrick Campus Center': ['recycling', 'compost', 'none'], "O'Shaughnessy Educational Center": ['recycling', 'compost', 'none'], "O'Shaughnessy-Frey Library Center": ['recycling', 'compost', 'none'], "O'Shaughnessy Science Hall": ['recycling', 'compost', 'none'], 'Owens Science Hall': ['recycling', 'compost', 'none'], "O'Shaughnessy Stadium": ['recycling', 'compost', 'none'], 'Schoenecker Center': ['LEED Gold'], 'Schoenecker Hall North': ['LEED Silver'], 'Center for Well-Being': ['recycling', 'compost', 'none'], 'Archbishop Ireland Memorial Library': ['recycling', 'compost', 'none']}


In [21]:
print(building_edges.columns)

Index(['Source', 'Target', 'Distance'], dtype='str')


## Build graph neighbors


 Create an empty dictionary to store graph neighbors  
 This will become our campus graph

 In graph terminology:
 - Nodes = buildings
 - Edges = valid movement connections between buildings

In [5]:
# Create an empty dictionary to store graph neighbors
# This will become our campus graph
#
# In graph terminology:
# - Nodes = buildings
# - Edges = valid movement connections between buildings
# Create empty neighbor graph
neighbors = {

    'Anderson Student Center': [
        "O'Shaughnessy Stadium",
        'Binz Refectory'
    ],

    'Anderson Parking Facility': [
        'Binz Refectory',
        'Brady Educational Center',
        "O'Shaughnessy Science Hall",
        'Schoenecker Center'
    ],

    'Binz Refectory': [
        'Anderson Student Center',
        'Anderson Parking Facility',
        'Brady Educational Center'
    ],

    'Brady Educational Center': [
        'Anderson Parking Facility',
        'Binz Refectory'
    ],

    'Center for Microgrid Research / Greenhouse': [
        "O'Shaughnessy Science Hall",
        'Owens Science Hall',
        'Schoenecker Center'
    ],

    'Frey Residence Hall': [
        'Murray-Herrick Campus Center',
        "O'Shaughnessy Educational Center",
        'Schoenecker Hall North'
    ],

    'Murray-Herrick Campus Center': [
        'Frey Residence Hall',
        "O'Shaughnessy Educational Center",
        "O'Shaughnessy-Frey Library Center"
    ],

    "O'Shaughnessy Educational Center": [
        'Frey Residence Hall',
        'Murray-Herrick Campus Center',
        "O'Shaughnessy-Frey Library Center"
    ],

    "O'Shaughnessy-Frey Library Center": [
        'Murray-Herrick Campus Center',
        "O'Shaughnessy Educational Center",
        'Center for Well-Being'
    ],

    "O'Shaughnessy Science Hall": [
        'Anderson Parking Facility',
        'Center for Microgrid Research / Greenhouse',
        'Owens Science Hall',
        'Schoenecker Center'
    ],

    'Owens Science Hall': [
        'Center for Microgrid Research / Greenhouse',
        "O'Shaughnessy Science Hall",
        'Schoenecker Center'
    ],

    "O'Shaughnessy Stadium": [
        'Anderson Student Center'
    ],

    'Schoenecker Center': [
        'Anderson Parking Facility',
        'Center for Microgrid Research / Greenhouse',
        "O'Shaughnessy Science Hall",
        'Owens Science Hall',
        'Center for Well-Being'
    ],

    'Schoenecker Hall North': [
        'Frey Residence Hall'
    ],

    'Center for Well-Being': [
        "O'Shaughnessy-Frey Library Center",
        'Schoenecker Center',
        'Archbishop Ireland Memorial Library'
    ],

    'Archbishop Ireland Memorial Library': [
        'Center for Well-Being'
    ]
}

print(neighbors)


{'Anderson Student Center': ["O'Shaughnessy Stadium", 'Binz Refectory'], 'Anderson Parking Facility': ['Binz Refectory', 'Brady Educational Center', "O'Shaughnessy Science Hall", 'Schoenecker Center'], 'Binz Refectory': ['Anderson Student Center', 'Anderson Parking Facility', 'Brady Educational Center'], 'Brady Educational Center': ['Anderson Parking Facility', 'Binz Refectory'], 'Center for Microgrid Research / Greenhouse': ["O'Shaughnessy Science Hall", 'Owens Science Hall', 'Schoenecker Center'], 'Frey Residence Hall': ['Murray-Herrick Campus Center', "O'Shaughnessy Educational Center", 'Schoenecker Hall North'], 'Murray-Herrick Campus Center': ['Frey Residence Hall', "O'Shaughnessy Educational Center", "O'Shaughnessy-Frey Library Center"], "O'Shaughnessy Educational Center": ['Frey Residence Hall', 'Murray-Herrick Campus Center', "O'Shaughnessy-Frey Library Center"], "O'Shaughnessy-Frey Library Center": ['Murray-Herrick Campus Center', "O'Shaughnessy Educational Center", 'Center fo

## CSP constraint


 Constraint function for the CSP

 This function checks whether two buildings are allowed  
 to have certain sustainability assignments at the same time.

 Parameters:  
 X = first building  
 x = sustainability assignment for X  

 Y = second building  
 y = sustainability assignment for Y  

 Example:  
 X = "Binz Refectory"  
 x = "compost"    

Y = "Anderson Student Center"  
y = "compost"  

In [6]:
def constraint(X, x, Y, y):

    if Y in neighbors[X]:

        if x == 'compost' and y == 'compost':
            return False

        if x == 'recycling' and y == 'recycling':
            return False

    return True



## CSP solver


Function to check whether a proposed assignment is valid  

Parameters:   
building = the current building we are trying to assign  
value = the sustainability ressource we want to assign  
assignment = the current partial CSP solution  

In [9]:
def is_valid(building, value, assignment):

    # Loop through every building already assigned
    for other_building in assignment:

        # Get the already-assigned value
        other_value = assignment[other_building]

        # Check the constraint function
        #
        # If the assignment violates a constraint:
        # return False immediately
        if not constraint(
            building,
            value,
            other_building,
            other_value
        ):
            return False

    # If no constraints are violated,
    # the assignment is valid
    return True


# Backtracking CSP Solver
#
# This function recursively tries assignments
# until it finds a complete valid solution.
#
# assignment = current partial solution

def backtrack(assignment):

    # Base Case:
    #
    # If every building has been assigned,
    # the CSP solution is complete
    if len(assignment) == len(variables):
        return assignment

    # Create a list of buildings
    # that still need assignments
    unassigned = [

        # Keep buildings NOT already assigned
        v for v in variables

        if v not in assignment
    ]

    # Select the next unassigned building
    #
    # This is a simple variable-selection strategy:
    # just pick the first remaining building
    current = unassigned[0]

    # Try every possible domain value

    for value in domains[current]:

        # Check whether this assignment is valid
        if is_valid(current, value, assignment):

            # Temporarily assign the value

            assignment[current] = value

            # Recursively continue solving
            result = backtrack(assignment)

            # If recursion found a valid solution:
            # return it immediately
            if result is not None:
                return result

            # Otherwise:
            # undo the assignment
            #
            # This is the "backtracking" step

            del assignment[current]

    # If no valid assignment works,
    # return None to backtrack further
    return None


# Start the CSP solver with an empty assignment
solution = backtrack({})

# Print the final valid CSP solution
print(solution)

{'Anderson Student Center': 'LEED Gold', 'Anderson Parking Facility': 'recycling', 'Binz Refectory': 'compost', 'Brady Educational Center': 'none', 'Center for Microgrid Research / Greenhouse': 'energy audit', 'Frey Residence Hall': 'LEED Platinum', 'Murray-Herrick Campus Center': 'recycling', "O'Shaughnessy Educational Center": 'compost', "O'Shaughnessy-Frey Library Center": 'none', "O'Shaughnessy Science Hall": 'compost', 'Owens Science Hall': 'recycling', "O'Shaughnessy Stadium": 'recycling', 'Schoenecker Center': 'LEED Gold', 'Schoenecker Hall North': 'LEED Silver', 'Center for Well-Being': 'recycling', 'Archbishop Ireland Memorial Library': 'compost'}


## Sustainability targets


In [ ]:
#Printing target buildings

# Do to high traffic, 
# food waste, dorm waste,  recycling demand, and energy usage


# If you don't answer correctly at these 
target_buildings = [
    'Anderson Student Center',
    'Binz Refectory',
    "O'Shaughnessy Science Hall",
    'Schoenecker Center'
]

print(target_buildings)



['Anderson Student Center', 'Binz Refectory', "O'Shaughnessy Science Hall", 'Schoenecker Center']


## Vocabulary words


In [9]:
try:
    with open("sustainability_words.txt", "r") as f:

        vocab_words = [
            line.strip().lower()
            for line in f.readlines()
        ]
except:
    vocab_words = [
        "compost",
        "recycle",
        "carbon",
        "energy"
    ]
print(vocab_words) 

['compost', 'recycle', 'carbon', 'energy']


## Trivia function


In [22]:
# Trivial function
import random
import json


with open("sustainability_trivia.json", "r") as f:
    trivia_data = json.load(f)

triv = trivia_data["true_false"]

def trivia_question():

    question = random.choice(triv)

    print("\nTrue or False?")
    print(question["question"])


    guess = input("True or False: ").lower().strip()
    print(f"You said {guess}")

    return guess == question["answer"]


In [5]:
if trivia_question():
    print("You are correct!")
else:
    print("Wrong answer!")

A bike repair station is located between ASC and O'Shaughnessy Stadium.
You are correct!


## Energy penalties


In [ ]:
energy_penalties = {}

# Setting the energy penalty for each building to 1
for building in variables:
    energy_penalties[building] = 1

# Loop through energy asset dataset
for row in energy_assets.itertuples():

    # check if the building exists in our graph
    if row.location_name in energy_penalties:

        # IF LEED platinum (most sustainable) Penalty = 0
        if "LEED Platinum" in str(row.designation_or_feature):
            energy_penalties[row.location_name] = 0
        # IF LEED Gold (highly sustainable) Penalty = 1
        elif "LEED Gold" in str(row.designation_or_feature):
            energy_penalties[row.location_name] = 1

        # IF anything else  Penalty = 3
        else:
            energy_penalties[row.location_name] = 3

print(energy_penalties)



{'Anderson Student Center': 1, 'Anderson Parking Facility': 1, 'Binz Refectory': 1, 'Brady Educational Center': 1, 'Center for Microgrid Research / Greenhouse': 3, 'Frey Residence Hall': 0, 'Murray-Herrick Campus Center': 1, "O'Shaughnessy Educational Center": 1, "O'Shaughnessy-Frey Library Center": 1, "O'Shaughnessy Science Hall": 1, 'Owens Science Hall': 1, "O'Shaughnessy Stadium": 1, 'Schoenecker Center': 1, 'Schoenecker Hall North': 3, 'Center for Well-Being': 1, 'Archbishop Ireland Memorial Library': 1}


## Heuristic


Heuristic function for A* search  

"How much work is left before reaching the goal?"  

Parameters:  
currernt_building = where the agent currently is  
remaining_targets = sustainability buildings that still need to be visited

In [13]:
def heuristic(current_building, remaining_targets):

    return len(remaining_targets)


## A* search


Movement cost function  

This determines how expensive it is to move into a building  

movement cost = base cost + energy penalty  

More sustainable buildings have LOWER penalties, so the AI is encouraged to travel through them.  

Goal: Find an efficient sustainable route through the campus graph while collecting sustainability resources.

In [21]:
def movement_cost(building):
    # IF building exists in energy_penalties use the value.
    # Otherwise use a default penalty of 1
    return 1 + energy_penalties.get(building, 1)


def a_star_search(start_building):

    # Convert target buildings into a set
    #
    # Sets allow faster lookup/checking
    targets = set(target_buildings)

    # Initial search state

    # frozenset() means:
    # no targets collected yet
    start_state = (
        start_building,
        frozenset()
    )

    # Priority queue for A*
    #
    # A* always expands the lowest estimated-cost state first
    queue = []

    # Push the starting node into the queue

    heapq.heappush(
        queue,
        (
            0,          #estimated total
            0,          #current sustainability cost
            0,          #player score
            start_state,
            [start_building],
            []
        )
    )

    # Keep track of explored states
    #
    # Prevents revisiting the same state repeatedly
    visited = set()

    # Main A* loop
    while queue:

        # Remove the lowest-cost item from the priority queue
        (
            estimated_total,
            current_cost,
            player_score,
            state,
            path,
            event_log
        ) = heapq.heappop(queue)

        # Unpack the state
        current_building, collected = state

        # Skip already-visited states
        if state in visited:
            continue

        # Mark current state as visited
        visited.add(state)

        # Goal Test
        #
        # If all targets have been collected:
        # return the solution
        if set(collected) == targets:
            print("\n===== FINAL RESULTS =====")

            print("\nFinal Path:")
            for i, building in enumerate(path, start=1):
                print(f"{i}. {building}")

            print(f"\nBuildings Visited: {len(path)}")

            print("\nTargets Collected:")
            for i, target in enumerate(collected, start=1):
                print(f"{i}. {target}")

            print(f"\nTotal Targets Collected: {len(collected)}")

            print(f"\nFinal Sustainability Cost: {current_cost}")

            print(f"Final Score: {player_score}")

            return " "
        # Explore neighboring buildings

        for neighbor in neighbors[current_building]:

            if neighbor in path:
                continue

            # Copy the collected targets

            new_collected = set(collected)

            # Add movement cost for entering the neighbor
            new_cost = current_cost + movement_cost(neighbor)

            # Copy the event log
            new_events = event_log.copy()

            new_score = player_score

            # Sustainability interaction
            #
            # If the building is a target building
            # AND has not already been collected
            if (
                neighbor in targets
                and neighbor not in collected
            ):

                print(f"Arrived at {neighbor}")

                # new_score = player_score

                # Trigger trivia/hangman challenge
                correct = trivia_question()

                # If correct:collect the target and reduce sustainability cost
                new_collected.add(neighbor)

                if correct:
                    # Reward
                    new_cost -= 2

                    new_score += 10
                    print(f"\nCorrect! +10 points")


                    new_events.append(f"Correct sustainability challenge at {neighbor}")
                else:
                    #Penalty
                    new_cost += 3

                    new_score -= 5
                    print(f"\nIncorrect! -5 points")
                    new_events.append(f"Incorrect sustainability challenge at {neighbor}")

                # Display current game stats
                print(f"Current Sustainability Cost: {new_cost}")
                print(f"Current Score: {new_score}")
                print(f"Collected Targets: {len(new_collected)}/{len(targets)}")
                print("-" * 40)
                
                # if correct:

                #     # Mark building as collected
                #     new_collected.add(neighbor)

                #     # Sustainability reward
                #     new_cost -= 2

                #     # Log the event
                #     new_events.append(
                #         f"Collected resource at {neighbor}"
                #     )

                # # If incorrect:
                # #
                # # - add sustainability penalty
                # else:

                #     new_cost += 3

                #     new_events.append(
                #         f"Failed challenge at {neighbor}"
                #     )

            # Remaining target buildings
            remaining = targets - new_collected

            # A* estimated total cost:
            #
            # f(n) = g(n) + h(n)
            #
            # g(n) = actual path cost
            # h(n) = heuristic estimate
            estimate = (
                new_cost
                + heuristic(neighbor, remaining)
            )

            # Create the new search state
            new_state = (
                neighbor,
                frozenset(new_collected)
            )

            # Push the new state into the priority queue
            heapq.heappush(
                queue,
                (
                    estimate,
                    new_cost,
                    new_score,
                    new_state,
                    path + [neighbor],
                    new_events
                )
            )

    # If no valid route is found:
    print("No valid route found.")
    return None

In [18]:
for building in target_buildings:
    print(building, neighbors[building])

Anderson Student Center ["O'Shaughnessy Stadium"]
Binz Refectory ['Anderson Parking Facility', 'Brady Educational Center']
O'Shaughnessy Science Hall ['Center for Microgrid Research / Greenhouse', 'Owens Science Hall', 'Schoenecker Center']
Schoenecker Center ['Anderson Parking Facility', 'Center for Microgrid Research / Greenhouse', "O'Shaughnessy Science Hall"]


In [19]:
print(neighbors["Center for Microgrid Research / Greenhouse"])

["O'Shaughnessy Science Hall", 'Owens Science Hall', 'Schoenecker Center']


## Run the game


In [19]:
possible_starts = [
    b for b in variables
    if b not in target_buildings
]

start_building = random.choice(possible_starts)

print("Start:", start_building)

result = a_star_search(start_building)

print(result)


Start: Archbishop Ireland Memorial Library
Arrived at Schoenecker Center
True or False
The Student Sustainability Leadership Program (SSLP) is only for students with environmental majors.
You said false
Correct! +10 points
Current Sustainability Cost: 2
Current Score: 10
Collected Targets: 1/4
----------------------------------------
Arrived at O'Shaughnessy Science Hall
True or False
At St. Thomas items that are placed in the green organics recycling bins are turned into compost that can be used in gardens and lawns.
You said true
Correct! +10 points
Current Sustainability Cost: 2
Current Score: 20
Collected Targets: 2/4
----------------------------------------
Arrived at Binz Refectory
True or False
A bike repair station is located between ASC and O'Shaughnessy Stadium.
You said true
Correct! +10 points
Current Sustainability Cost: 4
Current Score: 30
Collected Targets: 3/4
----------------------------------------
Arrived at Anderson Student Center
True or False
Keeping blinds closed